# Blinded pharmacogenomics study - 110 cases, 21 genes, 35 gene-drug pairs

Reproduces `claude_science_answers.csv` and the lookup tables from `cases_blinded.json`
and `cpic_guideline_corpus.json`, using the CPIC data API (api.cpicpgx.org) as the primary
source and the supplied guideline corpus where the API has no machine-readable rows.

All logic lives in `pgx_pipeline.py`; this notebook runs it and inspects the result.

## 1. Inputs

In [1]:
import json, csv, collections, os
import pgx_pipeline as P

print("cases:", len(P.CASES))
print("corpus genes:", len(P.CORPUS_GENES), "|", P.CORPUS_TAG)
print("gene-drug pairs:", len({(c['gene'], c['drug']) for c in P.CASES}))
collections.Counter(c['gene'] for c in P.CASES).most_common()

cases: 110
corpus genes: 21 | corpus v3.1 snapshot 2026-05-10T04:40:15Z
gene-drug pairs: 35


[('CYP2D6', 20), ('CYP2C19', 20), ('CYP2C9', 9), ('DPYD', 7), ('SLCO1B1', 6), ('UGT1A1', 6), ('G6PD', 6), ('CYP2B6', 4), ('TPMT', 4), ('HLA-B*15:02', 4), ('CYP3A5', 3), ('NUDT15', 3), ('HLA-B*57:01', 2), ('HLA-A*31:01', 2), ('HLA-B*58:01', 2), ('IFNL3', 2), ('VKORC1', 2), ('CYP4F2', 2), ('CFTR', 2), ('MT-RNR1', 2), ('RYR1', 2)]

## 2. Strand handling

CPIC stores allele-defining bases on the **forward genomic strand** (its `chromosomelocation`
HGVS), while several supplied genotype strings are written in **gene orientation**. For every
variant the literal bases are tested against the CPIC allele values at that position first, and
the reverse complement only if the literal bases are not valid alleles there. Below: the
relationship between the two conventions for each variant in the case set.

In [2]:
rows = []
for gene, rsids in P.RS_BY_GENE.items():
    for rs in rsids:
        for loc in P.LOC_BY_RS.get(rs, []):
            if loc['genesymbol'] != gene: continue
            rows.append((gene, rs, loc['name'], loc['chromosomelocation'], loc['genelocation'],
                         P.REF_BASE.get((gene, rs), '?')))
for r in sorted(rows):
    print(f"{r[0]:8s} {r[1]:12s} {r[2][:22]:24s} chrom {r[3][:28]:30s} gene {r[4][:18]:20s} fwd-ref {r[5]}")

CFTR     rs75527207   G551D/Class: III         chrom g.117587806G>A                 gene g.127023G>A          fwd-ref G
CYP2B6   rs2279343    18053A>G                 chrom g.41009358A>G                  gene g.23060A>G           fwd-ref A
CYP2B6   rs3745274    15631G>T                 chrom g.41006936G>T                  gene g.20638G>T           fwd-ref G
CYP2C19  rs12248560   -806C>T                  chrom g.94761900C>T                  gene g.4220C>T            fwd-ref C
CYP2C19  rs4244285    19154G>A                 chrom g.94781859G>A                  gene g.24179G>A           fwd-ref G
CYP2C9   rs1057910    42614A>C                 chrom g.94981296A>C                  gene g.48139A>C           fwd-ref A
CYP2D6   rs1065852    100C>T                   chrom g.42130692G>A                  gene g.5119C>T            fwd-ref G
CYP2D6   rs3892097    1847G>A                  chrom g.42128945C>T                  gene g.6866G>A            fwd-ref C
CYP3A5   rs776746     6981A>G           

## 3. Variant -> star allele (from CPIC allele definitions)

Built by the minimal-definition rule: the CPIC allele with the fewest defining positions that
carries the observed base. Note `CYP2D6*4` carries the IUPAC code `R` at rs1065852, i.e. the
*4 sub-alleles may or may not carry 100C>T - this is what allows the compound
rs3892097 C/T + rs1065852 G/A case to be resolved as *1/*4 in cis.

In [3]:
for (gene, rs, base), (name, note) in sorted(P.VAR2ALLELE.items()):
    print(f"{gene:8s} {rs:12s} {base} -> {name:34s} {note[:64]}")

CFTR     rs75527207   A -> G551D                              minimal CPIC allele definition carrying rs75527207=A (1 defining
CFTR     rs75527207   G -> ivacaftor non-responsive CFTR sequence minimal CPIC allele definition carrying rs75527207=G (93 definin
CYP2B6   rs2279343    A -> *1                                 minimal CPIC allele definition carrying rs2279343=A (48 defining
CYP2B6   rs2279343    G -> *4                                 minimal CPIC allele definition carrying rs2279343=G (1 defining 
CYP2B6   rs3745274    G -> *1                                 minimal CPIC allele definition carrying rs3745274=G (48 defining
CYP2B6   rs3745274    T -> *6                                 rs3745274 (516G>T) alone is shared by *6 and *9; the CPIC efavir
CYP2C19  rs12248560   C -> *38                                minimal CPIC allele definition carrying rs12248560=C (43 definin
CYP2C19  rs12248560   T -> *17                                minimal CPIC allele definition carrying rs122

## 4. Run the three-stage pipeline

In [4]:
rows, audit = P.main()

wrote claude_science_answers.csv: 110 rows
wrote case_audit.csv: 110 rows
wrote table_diplotype_to_phenotype.csv: 60 rows
wrote table_phenotype_to_recommendation.csv: 106 rows
wrote table_variant_to_allele.csv: 43 rows
wrote sources_and_versions.csv: 19 rows

110 cases; 0 abstentions
  LETHAL-CLASS 74597c5503ec CYP2D6/codeine: Ultrarapid Metabolizer
  LETHAL-CLASS ca9e14feca53 CYP2C19/clopidogrel: Poor Metabolizer
  LETHAL-CLASS 97d4e4559a13 CYP2C19/clopidogrel: Intermediate Metabolizer
  LETHAL-CLASS 97c3baebed4e CYP2C9/warfarin: Poor Metabolizer
  LETHAL-CLASS b94a43e574db DPYD/fluorouracil: Poor Metabolizer
  LETHAL-CLASS c57caa44a38a DPYD/fluorouracil: Intermediate Metabolizer
  LETHAL-CLASS 6b8d2db91001 DPYD/fluorouracil: Intermediate Metabolizer
  LETHAL-CLASS c6549d637d49 DPYD/capecitabine: Poor Metabolizer
  LETHAL-CLASS ae4a7215ae64 DPYD/capecitabine: Intermediate Metabolizer
  LETHAL-CLASS 08e18e809fcb TPMT/azathioprine: Poor Metabolizer
  LETHAL-CLASS f3dd19d8a337 NUDT15/thi

## 5. Answers

In [5]:
ans = list(csv.DictReader(open('claude_science_answers.csv')))
assert [a['case_id'] for a in ans] == [c['id'] for c in P.CASES]
print("rows:", len(ans), "| abstentions:", sum(1 for a in ans if a['abstained'] == 'true'))
for a in ans[:8]:
    print(f"{a['case_id']}  {a['diplotype'][:22]:24s} {a['phenotype'][:26]:28s} {a['recommendation'][:70]}")

rows: 110 | abstentions: 0
70dc9c9110f3  *4/*4                    Poor Metabolizer             codeine: Avoid codeine use because of possibility of diminished analge
b07cff8c99c0  *1/*4                    Intermediate Metabolizer     codeine: Use codeine label recommended age- or weight-specific dosing.
ce34b1a18a3d  *1/*1                    Normal Metabolizer           codeine: Use codeine label recommended age- or weight-specific dosing.
74597c5503ec  *1/*1xN                  Ultrarapid Metabolizer       codeine: Avoid codeine use because of potential for serious toxicity. 
3d0e97b86ad0  *1/*4                    Intermediate Metabolizer     codeine: Use codeine label recommended age- or weight-specific dosing.
f9e4a13551fd  *4/*4                    Poor Metabolizer             tamoxifen: Recommend alternative hormonal therapy such as an aromatase
9d75134030af  *1/*4                    Intermediate Metabolizer     tamoxifen: Consider hormonal therapy such as an aromatase inhibitor fo


## 6. Distribution of phenotypes and risk classes

In [6]:
print(collections.Counter(a['phenotype'] for a in audit).most_common())
print()
print(collections.Counter(a['lethal_class'] for a in audit).most_common())
print()
print("source of each recommendation:")
for k, v in collections.Counter(
        ('CPIC API' if a['recommendation_source'].startswith('CPIC') else 'guideline corpus')
        for a in audit).most_common():
    print(f"  {k:18s} {v}")

[('Intermediate Metabolizer', 22), ('Poor Metabolizer', 20), ('Normal Metabolizer', 20), ('Ultrarapid Metabolizer', 9), ('Rapid Metabolizer', 5), ('Deficient', 3), ('Poor Function', 2), ('Decreased Function', 2), ('Normal Function', 2), ('*15:02 positive', 2), ('*15:02 negative', 2), ('Normal', 2), ('*57:01 positive', 1), ('*57:01 negative', 1), ('*31:01 positive', 1), ('*31:01 negative', 1), ('*58:01 positive', 1), ('*58:01 negative', 1), ('Variable', 1), ('Favorable response genotype', 1), ('Unfavorable response genotype', 1), ('VKORC1 -1639 A/A (low warfarin dose requirement; no CPIC metaboliser phenotype assigned)', 1), ('VKORC1 -1639 G/G (reference; normal warfarin dose requirement; no CPIC metaboliser phenotype assigned)', 1), ('CYP4F2 *3/*3 (no CPIC phenotype assigned for CYP4F2)', 1), ('CYP4F2 *1/*1 (no CPIC phenotype assigned for CYP4F2)', 1), ('ivacaftor responsive in CF patients', 1), ('ivacaftor non-responsive in CF patients', 1), ('increased risk of aminoglycoside-induced 

## 7. Lethal-class cases

In [7]:
for a in audit:
    if a['lethal_class'] == 'lethal':
        print(f"{a['case_id']}  {a['gene']:12s} {a['drug']:22s} {a['phenotype'][:26]:28s}")
        print(f"             {a['lethal_rationale'][:104]}")

74597c5503ec  CYP2D6       codeine                Ultrarapid Metabolizer      
             CYP2D6 ultrarapid metaboliser: excess morphine formation from codeine has caused fatal respiratory depre
ca9e14feca53  CYP2C19      clopidogrel            Poor Metabolizer            
             Reduced clopidogrel activation after ACS/PCI: markedly increased risk of stent thrombosis, myocardial in
97d4e4559a13  CYP2C19      clopidogrel            Intermediate Metabolizer    
             Reduced clopidogrel activation after ACS/PCI: markedly increased risk of stent thrombosis, myocardial in
97c3baebed4e  CYP2C9       warfarin               Poor Metabolizer            
             CYP2C9 poor metaboliser: standard warfarin loading causes over-anticoagulation and potentially fatal hae
b94a43e574db  DPYD         fluorouracil           Poor Metabolizer            
             Near-complete DPD deficiency: standard fluoropyrimidine dosing causes early-onset, potentially fatal muc
c57caa44a38a  D

## 8. Cases carrying an explicit caveat

In [8]:
for a in audit:
    if a['caveat']:
        print(f"{a['case_id']} {a['gene']}/{a['drug']}: {a['caveat'][:150]}")

ce34b1a18a3d CYP2D6/codeine: A negative SNV panel cannot exclude untested alleles (or, for CYP2D6, gene deletions/duplications); CPIC reports such a result as the reference diplot
74597c5503ec CYP2D6/codeine: N is not resolved by the assay; every xN>=2 gives activity score >=3.0, so the CPIC phenotype (ultrarapid metabolizer) is the same. Phenotype looked u
987d7bd14933 CYP2D6/tamoxifen: A negative SNV panel cannot exclude untested alleles (or, for CYP2D6, gene deletions/duplications); CPIC reports such a result as the reference diplot
61741d10f238 CYP2D6/tamoxifen: N is not resolved by the assay; every xN>=2 gives activity score >=3.0, so the CPIC phenotype (ultrarapid metabolizer) is the same. Phenotype looked u
3ac7c0b0f795 CYP2D6/ondansetron: A negative SNV panel cannot exclude untested alleles (or, for CYP2D6, gene deletions/duplications); CPIC reports such a result as the reference diplot
01166e2ea9ad CYP2D6/ondansetron: N is not resolved by the assay; every xN>=2 gives activity 